In [0]:
from datetime import date, datetime
from uuid import uuid4

from pyspark.sql.functions import (
    col, count, countDistinct, current_date, current_timestamp,
    explode, from_json, lit, sha2, size,
)
from pyspark.sql.types import (
    ArrayType, LongType, StringType, StructField, StructType, TimestampType,
)

dbutils.widgets.text("input_path", "", "Input JSON file or directory")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load type")
dbutils.widgets.text("batch_id", "", "Stable batch ID")
dbutils.widgets.text("source_extracted_at", "", "Source snapshot time (ISO 8601)")
dbutils.widgets.text("window_start", "", "Incremental window start (YYYY-MM-DD)")
dbutils.widgets.text("window_end", "", "Incremental window end (YYYY-MM-DD)")

params = {
    name: dbutils.widgets.get(name).strip()
    for name in ("input_path", "load_type", "batch_id",
                 "source_extracted_at", "window_start", "window_end")
}

if not params["input_path"] or not params["batch_id"]:
    raise ValueError("input_path and batch_id are required")
if params["load_type"] not in {"full", "incremental"}:
    raise ValueError("load_type must be 'full' or 'incremental'")
if params["load_type"] == "full" and (params["window_start"] or params["window_end"]):
    raise ValueError("Full loads should not have incremental window dates")
if bool(params["window_start"]) != bool(params["window_end"]):
    raise ValueError("Provide both window dates, or leave both blank")
for window_date in (params["window_start"], params["window_end"]):
    if window_date:
        date.fromisoformat(window_date)

snapshot_timestamp = datetime.fromisoformat(
    params["source_extracted_at"].replace("Z", "+00:00")
)
if snapshot_timestamp.tzinfo is None:
    raise ValueError("source_extracted_at must include a timezone")

print("Parameters valid:", params)

In [0]:
wrapper_schema = StructType([
    StructField("load_type", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("window_start", StringType(), True),
    StructField("window_end", StringType(), True),
    StructField("records", ArrayType(StringType()), True),
])
movie_id_schema = StructType([StructField("id", LongType(), True)])

files_df = (
    spark.read
    .option("multiLine", True)
    .schema(wrapper_schema)
    .json(params["input_path"])
)

if files_df.limit(1).count() == 0:
    raise ValueError(f"No input files found at: {params['input_path']}")

display(
    files_df.select(
        col("_metadata.file_path").alias("source_file"),
        "load_type",
        "record_count",
        size("records").alias("actual_record_count"),
        "window_start",
        "window_end",
    )
)

records_df = (
    files_df
    .select(col("_metadata.file_path").alias("source_file"), explode("records").alias("raw_payload"))
    .withColumn("movie_id", from_json("raw_payload", movie_id_schema).getField("id"))
)

print("Movie records read:", records_df.count())
print("Records missing a valid movie ID:", records_df.filter(col("movie_id").isNull()).count())
print(
    "Duplicate movie IDs in this input:",
    records_df.groupBy("movie_id").count()
    .filter((col("movie_id").isNotNull()) & (col("count") > 1))
    .count(),
)

In [0]:
duplicate_summary = (
    records_df
    .groupBy("movie_id")
    .agg(
        count("*").alias("row_count"),
        countDistinct(sha2(col("raw_payload"), 256)).alias("distinct_payloads"),
    )
    .filter(col("row_count") > 1)
    .orderBy(col("row_count").desc(), col("movie_id"))
)
display(duplicate_summary)

# A movie should not have conflicting versions within the same source batch.
conflicting_movies = (
    records_df
    .groupBy("movie_id")
    .agg(countDistinct(sha2(col("raw_payload"), 256)).alias("payload_versions"))
    .filter(col("payload_versions") > 1)
)
if conflicting_movies.limit(1).count() > 0:
    display(conflicting_movies)
    raise ValueError(
        "Conflicting payloads found for a movie in this batch; "
        "do not deduplicate them without resolving the source conflict."
    )

# Identical duplicates are safe to collapse; there is one batch per notebook run.
bronze_source_df = (
    records_df
    .dropDuplicates(["movie_id"])
    .select(
        "movie_id",
        lit(params["load_type"]).alias("load_type"),
        lit(params["batch_id"]).alias("batch_id"),
        "source_file",
        lit(snapshot_timestamp).cast("timestamp").alias("source_extracted_at"),
        lit(params["window_start"] or None).cast("date").alias("window_start"),
        lit(params["window_end"] or None).cast("date").alias("window_end"),
        current_date().alias("ingestion_date"),
        current_timestamp().alias("load_timestamp"),
        "raw_payload",
    )
)

print("Bronze input rows after deduplication:", bronze_source_df.count())
bronze_source_df.printSchema()
display(bronze_source_df.select("movie_id", "batch_id", "source_file"))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.bronze_movie_records (
    movie_id BIGINT, load_type STRING, batch_id STRING, source_file STRING,
    source_extracted_at TIMESTAMP, window_start DATE, window_end DATE,
    ingestion_date DATE, load_timestamp TIMESTAMP, raw_payload STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.pipeline_execution_logs (
    run_id STRING, layer STRING, load_type STRING, input_parameter STRING,
    batch_id STRING, start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
    rows_inserted BIGINT, rows_updated BIGINT, rows_quarantined BIGINT,
    error_details STRING, load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("SHOW TABLES IN workspace.tmdb_lakehouse").show(truncate=False)

In [0]:
bronze_table = "workspace.tmdb_lakehouse.bronze_movie_records"
logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"
run_id = str(uuid4())
run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
rows_inserted = rows_updated = rows_quarantined = 0

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])


def write_run_log(status, error_details=None):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    row = [(
        run_id, "Raw-to-Bronze", params["load_type"], params["input_path"],
        params["batch_id"], run_started_at, finished_at, status,
        rows_inserted, rows_updated, rows_quarantined, error_details, finished_at,
    )]
    (
        spark.createDataFrame(row, schema=log_schema)
        .write.format("delta").mode("append").saveAsTable(logs_table)
    )


try:
    source_row_count = bronze_source_df.count()
    rows_inserted = bronze_source_df.join(
        spark.table(bronze_table).select("batch_id", "movie_id"),
        ["batch_id", "movie_id"], "left_anti",
    ).count()

    bronze_source_df.createOrReplaceTempView("bronze_merge_source")
    spark.sql(f"""
        MERGE INTO {bronze_table} AS target
        USING bronze_merge_source AS source
        ON target.batch_id = source.batch_id AND target.movie_id = source.movie_id
        WHEN NOT MATCHED THEN INSERT *
    """)

    persisted_batch_count = (
        spark.table(bronze_table).filter(col("batch_id") == params["batch_id"]).count()
    )
    if persisted_batch_count != source_row_count:
        raise RuntimeError(
            f"Expected {source_row_count} Bronze rows for this batch, "
            f"found {persisted_batch_count}"
        )

    write_run_log("SUCCESS")
    print(f"Bronze merge succeeded. Rows inserted: {rows_inserted}; rows updated: 0")
    print(f"Rows now stored for this batch: {persisted_batch_count}")

except Exception as exc:
    try:
        write_run_log("FAILURE", str(exc)[:4000])
    except Exception as log_exc:
        print(f"Could not write the failure log: {log_exc}")
    raise

In [0]:
spark.sql("""
SELECT batch_id, status, rows_inserted, rows_updated, rows_quarantined
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Raw-to-Bronze'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)